# Stipple G2 smoke test

1M points from a 3-component Gaussian mixture, colored by cluster label.

Expected: three visually distinct color blobs, pan/zoom responsive, FPS > 30 reported.

In [ ]:
import numpy as np
from stipple import Stipple
import stipple
print(f"stipple {stipple.__version__}")

In [ ]:
rng = np.random.default_rng(42)
n = 1_000_000
n_per = n // 3
labels = np.repeat(['cluster_a', 'cluster_b', 'cluster_c'], n_per)
# Pad to n if not divisible
if labels.shape[0] < n:
    labels = np.concatenate([labels, np.full(n - labels.shape[0], 'cluster_a')])

centers = np.array([[-3.0, 0.0], [3.0, 0.0], [0.0, 3.0]], dtype=np.float32)
x = np.empty(n, dtype=np.float32)
y = np.empty(n, dtype=np.float32)
for k in range(3):
    sl = slice(k * n_per, (k + 1) * n_per)
    x[sl] = centers[k, 0] + rng.standard_normal(n_per).astype(np.float32) * 0.8
    y[sl] = centers[k, 1] + rng.standard_normal(n_per).astype(np.float32) * 0.8
# Fill any tail
if n_per * 3 < n:
    x[n_per*3:] = centers[0, 0]
    y[n_per*3:] = centers[0, 1]
print(f"prepared {n:,} points across {len(np.unique(labels))} clusters")

In [ ]:
w = Stipple(x=x, y=y, color=labels)
w

In [ ]:
import time
for _ in range(60):
    if w.last_fps:
        break
    time.sleep(0.1)
print(f"status            : {w.status}")
print(f"rows_received     : {w.rows_received:,}")
print(f"bytes_received    : {w.bytes_received:,}")
print(f"categories        : {w.color_categories}")
print(f"FPS               : {w.last_fps:.1f}")
print(f"avg frame ms      : {w.avg_frame_ms:.2f}")